## Basic RAG with Ollama + ChromaDB

This notebook demonstrates a basic Retrieval-Augmented Generation (RAG) pipeline
using local Ollama models and ChromaDB as the vector store.


In [1]:
import os
import sys

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IN_COLAB or IN_KAGGLE:
    !pip install git+https://github.com/saikrishna1729/reliablerag.git@rag_pipeline/jithu -q

In [2]:
import os

from langchain_core.documents import Document

from reliablerag.chain import build_rag_chain
from reliablerag.env import load_secrets
from reliablerag.providers import create_embeddings, create_llm
from reliablerag.retriever import build_vector_store, get_retriever

### 1. Configuration

Model names and paths are loaded from `.env`. Fallback defaults are used if not set.

In [ ]:
load_secrets()

PROVIDER           = os.environ["PROVIDER"]
EMBEDDING_MODEL    = os.environ["EMBEDDING_MODEL"]
LLM_MODEL          = os.environ["LLM_MODEL"]
CHROMA_PERSIST_DIR = os.environ["CHROMA_PERSIST_DIR"]

print(f"Provider        : {PROVIDER}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"LLM model       : {LLM_MODEL}")
print(f"Chroma dir      : {CHROMA_PERSIST_DIR}")

In [ ]:
embeddings = create_embeddings(PROVIDER, EMBEDDING_MODEL)
llm = create_llm(PROVIDER, LLM_MODEL)

### 2. Prepare the Knowledge Base

In [4]:
sample_docs = [
    Document(
        page_content="ReliableRAG is a specialized framework designed to eliminate LLM hallucinations using strict verification pipelines.",
        metadata={"source": "project_doc.txt"},
    ),
    Document(
        page_content="The core architecture of ReliableRAG uses a two-step validation: context relevance checking and response alignment.",
        metadata={"source": "architecture_doc.txt"},
    ),
    Document(
        page_content="Ollama runs LLMs locally on your machine using optimized quantization frameworks like llama.cpp.",
        metadata={"source": "ollama_doc.txt"},
    ),
]

### 3. Build the Vector Store

In [5]:
print("Embedding documents and indexing into Chroma DB...")
vector_store = build_vector_store(
    sample_docs,
    embeddings=embeddings,
    persist_directory=CHROMA_PERSIST_DIR,
)
retriever = get_retriever(vector_store)
print("Done.")

Embedding documents and indexing into Chroma DB...
Done.


### 4. Build and Run the RAG Chain

In [6]:
rag_chain = build_rag_chain(retriever, llm=llm)

query = "What is ReliableRAG and how does its architecture ensure validation?"
print(f"Query: {query}\n")

response = rag_chain.invoke(query)
print(f"Response:\n{response}")

Query: What is ReliableRAG and how does its architecture ensure validation?

Response:
ReliableRAG is a specialized framework designed to eliminate LLM hallucinations using strict verification pipelines. Its architecture ensures validation through a two-step process consisting of context relevance checking and response alignment.
